In [ ]:
import numpy as np
import random as rnd
from deap import base, creator, tools, algorithms

In [ ]:
import numpy as np
import random as rnd
from deap import base, creator, tools, algorithms

N_DISKS = 4
N_PEGS = 3
POP_SIZE = 300
NGEN = 200
CXPB = 0.7
MUTPB = 0.3
TOURNAMENT_SIZE = 3
MAX_STEPS = 100

creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

def init_individual(icls, n_disks, n_pegs, max_steps):
    moves = []
    for _ in range(max_steps):
        from_peg = rnd.randint(0, n_pegs - 1)
        to_peg = rnd.randint(0, n_pegs - 1)
        while to_peg == from_peg:
            to_peg = rnd.randint(0, n_pegs - 1)
        moves.append((from_peg, to_peg))
    return icls(moves)

def evaluate(individual, n_disks, n_pegs):
    pegs = [list(range(n_disks, 0, -1))] + [[] for _ in range(n_pegs - 1)]
    penalty = 0
    for from_peg, to_peg in individual:
        if not pegs[from_peg]:
            penalty += 1
            continue
        disk = pegs[from_peg][-1]
        if pegs[to_peg] and pegs[to_peg][-1] < disk:
            penalty += 1
            continue
        pegs[from_peg].pop()
        pegs[to_peg].append(disk)
    goal_peg = pegs[-1]
    if len(goal_peg) == n_disks and goal_peg == list(range(n_disks, 0, -1)):
        return (penalty,)
    disks_on_goal = len(goal_peg)
    correct_order = sum(1 for i, d in enumerate(goal_peg) if d == n_disks - i)
    return (penalty + (n_disks - disks_on_goal) * 10 + (n_disks - correct_order) * 5,)

def mutate_moves(individual, n_pegs, indpb=0.1):
    for i in range(len(individual)):
        if rnd.random() < indpb:
            from_peg = rnd.randint(0, n_pegs - 1)
            to_peg = rnd.randint(0, n_pegs - 1)
            while to_peg == from_peg:
                to_peg = rnd.randint(0, n_pegs - 1)
            individual[i] = (from_peg, to_peg)
    return (individual,)

def cx_two_point(ind1, ind2):
    size = min(len(ind1), len(ind2))
    cx1 = rnd.randint(1, size - 1)
    cx2 = rnd.randint(1, size - 1)
    if cx1 > cx2:
        cx1, cx2 = cx2, cx1
    ind1[cx1:cx2], ind2[cx1:cx2] = ind2[cx1:cx2], ind1[cx1:cx2]
    return ind1, ind2

toolbox = base.Toolbox()
toolbox.register("individual", init_individual, creator.Individual, N_DISKS, N_PEGS, MAX_STEPS)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", evaluate, n_disks=N_DISKS, n_pegs=N_PEGS)
toolbox.register("mate", cx_two_point)
toolbox.register("mutate", mutate_moves, n_pegs=N_PEGS, indpb=0.1)
toolbox.register("select", tools.selTournament, tournsize=TOURNAMENT_SIZE)

stats = tools.Statistics(lambda ind: ind.fitness.values[0])
stats.register("min", np.min)
stats.register("avg", np.mean)
stats.register("max", np.max)

hof = tools.HallOfFame(1)

In [ ]:
import numpy as np
from deap import base, creator

creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

In [ ]:
def evaluate(individual):
    pegs = [list(range(N_DISKS, 0, -1)), [], []] 
    steps = 0
    penalty = 0
    
    for move in individual:
        if steps >= MAX_STEPS:
            penalty += 1000
            break
        from_peg, to_peg = move
        
        if not (0 <= from_peg < N_PEGS and 0 <= to_peg < N_PEGS):
            penalty += 10
            continue
        if not pegs[from_peg]:
            penalty += 10
            continue
        if pegs[to_peg] and pegs[from_peg][-1] > pegs[to_peg][-1]:
            penalty += 10
            continue
        
        disk = pegs[from_peg].pop()
        pegs[to_peg].append(disk)
        steps += 1
        
        if pegs[2] == list(range(N_DISKS, 0, -1)):
            break
    
    remaining = len(pegs[0]) + len(pegs[1])
    wrong_order = sum(1 for p in pegs for i in range(len(p)-1) if p[i] < p[i+1])
    
    fitness = steps + remaining * 10 + wrong_order * 5 + penalty
    return (fitness,)

In [ ]:
import random as rnd

def cxTwoPointCopy(ind1, ind2):
    size = min(len(ind1), len(ind2))
    cxpoint1 = rnd.randint(1, size)
    cxpoint2 = rnd.randint(1, size - 1)
    if cxpoint2 >= cxpoint1:
        cxpoint2 += 1
    else:
        cxpoint1, cxpoint2 = cxpoint2, cxpoint1

    ind1[cxpoint1:cxpoint2], ind2[cxpoint1:cxpoint2] = ind2[cxpoint1:cxpoint2], ind1[cxpoint1:cxpoint2]
    return ind1, ind2

In [ ]:
import random as rnd

def mutShuffleIndexes(individual, indpb):
    size = len(individual)
    for i in range(size):
        if rnd.random() < indpb:
            swap_idx = rnd.randint(0, size - 1)
            individual[i], individual[swap_idx] = individual[swap_idx], individual[i]
    return individual,

In [ ]:
toolbox.register("select", tools.selTournament, tournsize=TOURNAMENT_SIZE)

In [ ]:
import random as rnd
import numpy as np
from deap import base, creator, tools

N_PEGS = 3
N_DISKS = 4
MAX_STEPS = 2 * N_DISKS * N_DISKS

creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

def initIndividual(icls, n_moves):
    moves = [(rnd.randint(0, N_PEGS-1), rnd.randint(0, N_PEGS-1)) for _ in range(n_moves)]
    return icls(moves)

toolbox = base.Toolbox()
toolbox.register("individual", initIndividual, creator.Individual, n_moves=MAX_STEPS)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

In [ ]:
from deap import base, tools
import numpy as np

TOURNAMENT_SIZE = 3

def evaluate(individual):
    # Placeholder for actual evaluation logic
    return (0,)

def cxTwoPointCopy(ind1, ind2):
    # Placeholder for actual crossover logic
    return ind1, ind2

toolbox = base.Toolbox()
toolbox.register("evaluate", evaluate)
toolbox.register("mate", cxTwoPointCopy)
toolbox.register("mutate", tools.mutShuffleIndexes, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=TOURNAMENT_SIZE)

In [ ]:
import random as rnd
import numpy as np
from deap import base, creator, tools, algorithms

N_DISKS = 5
POP_SIZE = 300
CXPB = 0.7
MUTPB = 0.2
NGEN = 200
MAX_MOVES = 2 * (2 ** N_DISKS)

creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()
toolbox.register("attr_move", rnd.randint, 0, 2)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_move, n=MAX_MOVES)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def eval_hanoi(individual):
    pegs = [list(range(N_DISKS, 0, -1)), [], []]
    penalty = 0
    moves = 0
    for fr, to in individual:
        if pegs[2] == list(range(N_DISKS, 0, -1)):
            break
        if fr == to:
            penalty += 10
            continue
        if not pegs[fr]:
            penalty += 10
            continue
        if pegs[to] and pegs[fr][-1] > pegs[to][-1]:
            penalty += 10
            continue
        disk = pegs[fr].pop()
        pegs[to].append(disk)
        moves += 1
    if pegs[2] != list(range(N_DISKS, 0, -1)):
        penalty += 1000 * (N_DISKS - len(pegs[2]))
    return (penalty + moves,)

toolbox.register("evaluate", eval_hanoi)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutUniformInt, low=0, up=2, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)

def main():
    rnd.seed(42)
    np.random.seed(42)
    
    pop = toolbox.population(n=POP_SIZE)
    hof = tools.HallOfFame(1)
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("std", np.std)
    stats.register("min", np.min)
    stats.register("max", np.max)
    
    algorithms.eaSimple(pop, toolbox, cxpb=CXPB, mutpb=MUTPB, ngen=NGEN,
                        stats=stats, halloffame=hof, verbose=True)
    
    best = hof[0]
    print(f"Best fitness: {best.fitness.values[0]}")
    print(f"Solution length: {len(best)}")
    
    pegs = [list(range(N_DISKS, 0, -1)), [], []]
    for i, (fr, to) in enumerate(best):
        if pegs[2] == list(range(N_DISKS, 0, -1)):
            break
        if pegs[fr] and (not pegs[to] or pegs[fr][-1] < pegs[to][-1]):
            disk = pegs[fr].pop()
            pegs[to].append(disk)
            print(f"Move {i+1}: Disk {disk} from Peg {fr+1} to Peg {to+1}")
    
    return pop, stats, hof

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("avg", np.mean)
stats.register("std", np.std)
stats.register("min", np.min)
stats.register("max", np.max)

In [ ]:
import matplotlib.pyplot as plt

def plot_stats(stats_log):
    gen = stats_log.select("gen")
    fit_mins = stats_log.select("min")
    fit_avgs = stats_log.select("avg")
    
    plt.figure(figsize=(10, 5))
    plt.plot(gen, fit_mins, "b-", label="Minimum Fitness")
    plt.plot(gen, fit_avgs, "r-", label="Average Fitness")
    plt.xlabel("Generation")
    plt.ylabel("Fitness")
    plt.title("Tower of Hanoi GA Convergence")
    plt.legend()
    plt.grid(True)
    plt.show()